In [ ]:
import pandas as pd
from collections import defaultdict

# STEP 1: Load CSVs directly from file paths
ref_df = pd.read_csv("cleaned_data_with_salts.csv")
med_df = pd.read_csv("Doctor_medicines_filled_generated.csv")

# STEP 2: Tokenize salts
def tokenize(s):
    return sorted(set(str(s).lower()
        .replace("mg", "")
        .replace("(", "").replace(")", "")
        .replace("/", " ").replace(",", " ")
        .split()))

ref_df["salt_tokens"] = ref_df["Salts"].apply(tokenize)
med_df["salt_tokens"] = med_df["Salts"].apply(tokenize)

# STEP 3: Build a hash map from salt token sets to reference medicines
salt_hash_map = defaultdict(list)
for _, row in ref_df.iterrows():
    key = tuple(sorted(row["salt_tokens"]))
    salt_hash_map[key].append((row["Name"], row["salt_tokens"], row["Price"]))

# STEP 4: Function to find cheaper alternatives with partial match
def find_cheaper(row):
    base_tokens = set(row["salt_tokens"])
    alternatives = []

    for ref_salts, meds in salt_hash_map.items():
        ref_tokens = set(ref_salts)
        if base_tokens & ref_tokens:  # if any salt matches
            for med_name, med_salts, price in meds:
                if price < row["Price"]:
                    alternatives.append((med_name, med_salts, price))

    return alternatives

# STEP 5: Apply and label
med_df["cheaper_alternatives"] = med_df.apply(find_cheaper, axis=1)
med_df["has_cheaper_alternative"] = med_df["cheaper_alternatives"].apply(lambda x: len(x) > 0)

# STEP 6: Save result
filtered_df = med_df[med_df["has_cheaper_alternative"]]
filtered_df.to_csv("expensive_afterHashing.csv", index=False)

# STEP 7: Report summary
total = len(med_df)
with_alt = len(filtered_df)

print(f"\n✅ Using hash maps and allowing partial/similar salt matches, the system found:\n")
print(f"{with_alt:,} medicines out of {total:,} have at least one cheaper alternative.")
print("📦 Result saved to 'expensive_afterHashing.csv'")
